# Lab 03 (Colab) - Sampling and Post-Training, by hand

**Chapter 2 (Understanding Foundation Models): Post-Training and Sampling** | Environment: **Google Colab, free T4 GPU** | Time: about 2.5 hours (do this notebook before `lab03_cluster.ipynb`)

A model does not output an answer; it outputs a probability for every token in its vocabulary, and a **sampling** strategy picks one. Chapter 2 argues that this one step explains a lot of model behavior: creativity, inconsistency, even hallucination. In this notebook you implement temperature, top-k, and top-p yourself on real logits and prove your code matches Hugging Face's `generate()` (the `transformers` function that writes text with a model). Then you measure how sampling settings change output diversity and the consistency of answers to Larkspur questions. Finally you compare a **base** model with its **post-trained** (instruct) sibling to see what post-training adds.

Models: `Qwen/Qwen3-1.7B` (post-trained) and `Qwen/Qwen3-1.7B-Base` (pretrained only). Both fit on a T4 at the same time (about 3.5 GB each in float16).

**Before you start**
- **Runtime > Change runtime type > T4 GPU > Save.**
- Model downloads are about 4 GB each.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Logit** | The raw score the model gives each token in its vocabulary for "what comes next" (from Lab 01). Higher means more likely. | Every sampling method in this lab works on these scores. |
| **Softmax** | The formula that turns logits into probabilities that add up to 100% (from Lab 01). | You write it yourself in A.1. |
| **Probability distribution** | The full list of probabilities, one per token, for the next token. | "Sharp" (one token very likely) vs "flat" (many tokens similar) is what temperature changes. |
| **Sampling** | Picking the next token at random, weighted by its probability. | It's why a model can give a different answer each time you ask. |
| **Greedy decoding** | Always picking the single most likely token instead of sampling (from Lab 01). | Same input, same output (in theory; the cluster notebook tests it). |
| **Temperature** | A setting that sharpens (below 1) or flattens (above 1) the distribution before sampling. | The main knob for "creative" vs "consistent" output. |
| **Top-k / top-p** | Filters that throw away unlikely tokens before sampling: keep the `k` best, or keep the smallest set that adds up to probability `p`. | They stop the model from occasionally picking a nonsense token. |
| **Entropy** | A number (in bits) for how spread out a distribution is. 0 bits means certain; higher means less sure. | You use it to measure how confident a model is. |
| **Tensor** | PyTorch's name for a list or grid of numbers that can live on the GPU. | Logits and token IDs are tensors in the code. |
| **n-gram, distinct-n** | An n-gram is a run of n words in a row. Distinct-n is the share of n-grams that are unique across several answers. | It measures how varied a batch of answers is (B.1). |
| **Hallucination** | A confident answer that is made up. | Part B shows when it happens and whether temperature affects it. |
| **Grounded** | The model was given the facts it needs in its context (for example, a document). | Grounding is the main defense against hallucination. |
| **Base vs post-trained (instruct) model** | A base model only learned to continue text. A post-trained model was then trained further to follow instructions and chat (from Lab 01). | B.3 compares the two directly. |
| **SFT, preference finetuning** | The two post-training steps in Chapter 2: supervised finetuning on example conversations, then training on which answers people prefer (RLHF, DPO). | They explain the differences you see in B.3. |

## Setup

The first cell installs the course helper package and an up-to-date `transformers` (Hugging Face's library for running open models; `accelerate` helps it place the model on the GPU). The second cell loads the instruct model and defines three helpers you'll use all the way through:
- `chat_inputs()` turns chat messages into token IDs, using the model's **chat template** (the special-token format from Lab 01 that marks the system, user, and assistant turns). It also turns off Qwen3's "thinking" mode, where the model writes out its reasoning before answering.
- `next_logits()` runs the model once and returns its scores (logits) for the first token of the answer.
- `show_top()` prints the most likely next tokens as a small table.

In PyTorch, numbers live in **tensors**: lists or grids of numbers that can sit on the GPU.

In [ ]:
!pip -q install "itec632 @ git+https://github.com/tyler-schunk/itec632-course#subdirectory=image"
!pip -q install -U transformers accelerate

In [ ]:
import re, math, time
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from itec632.grading import LabChecker

ck = LabChecker("lab03", part="colab")
gpu = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if gpu else "NONE - use Runtime > Change runtime type > T4 GPU")
ck.check("setup_gpu", gpu, 1, "Select a T4 GPU (Runtime > Change runtime type), then Runtime > Run all.")

INSTRUCT_ID = "Qwen/Qwen3-1.7B"
DTYPE = torch.float16   # if you see "nan"/"inf" errors, change to torch.float32 (the T4 has no bfloat16 support).
                        # float32 doubles memory: if Colab runs out of memory at B.3, Runtime > Restart session, Run all.
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_ID)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_ID, dtype=DTYPE, device_map="auto")
model.eval()
print("loaded", INSTRUCT_ID, "on", model.device)


def chat_inputs(messages, tok=None, mdl=None):
    """Chat messages -> tensors. enable_thinking=False turns off Qwen3's reasoning phase for this lab."""
    tok, mdl = tok or tokenizer, mdl or model
    return tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                   return_tensors="pt", return_dict=True).to(mdl.device)


def next_logits(messages):
    """Raw logits (float32, 1-D over the vocabulary) for the first answer token."""
    with torch.no_grad():
        return model(**chat_inputs(messages)).logits[0, -1, :].float()


def show_top(logits, k=8, title=""):
    probs = torch.softmax(logits, dim=-1)
    top_p, top_i = torch.topk(probs, k)
    rows = [{"token": repr(tokenizer.decode([int(i)])), "logit": round(logits[i].item(), 2),
             "prob_%": round(p.item() * 100, 2)} for p, i in zip(top_p, top_i)]
    if title:
        print(title)
    display(pd.DataFrame(rows))

# Part A - Guided practice: build a sampler

## A.1 Logits and softmax

For each position the model produces one raw score, a **logit**, per vocabulary token. Chapter 2 (**"Sampling Fundamentals"**) explains that **softmax** turns logits into probabilities: `p_i = exp(x_i) / sum_j exp(x_j)`. Subtracting the largest logit first does not change the result but avoids **overflow** (`exp()` of a large number is too big for the computer to store).

**Reading the output:** `logits shape` is one score per vocabulary token, so that number is the vocabulary size. The table lists the 8 most likely first tokens of the answer: `token` is the piece of text (in quotes so you can see spaces), `logit` is its raw score, and `prob_%` is its probability after softmax.

In [ ]:
PROMPT = [{"role": "system", "content": "You are Larkspur Ridge IT's help-desk assistant."},
          {"role": "user", "content": "Give one word that describes a great IT technician. Reply with just the word."}]
logits = next_logits(PROMPT)
print("logits shape:", tuple(logits.shape), " (one score per vocabulary token)")
print(f"min {logits.min().item():.1f}, max {logits.max().item():.1f}")
show_top(logits, title="Top candidates at temperature 1:")

**TODO A.1:** implement softmax yourself.

In [ ]:
def my_softmax(x):
    """Softmax over a 1-D tensor of logits. Works when some entries are -inf (they get probability 0)."""
    # TODO: use the formula in the text: subtract x.max() first, exponentiate with torch.exp,
    #       then divide by the sum of the exponentials.
    return None


p_mine = my_softmax(logits)
print(None if p_mine is None else float(p_mine.sum()))

In [ ]:
ck.check("A1_softmax", p_mine is not None and torch.allclose(p_mine, torch.softmax(logits, -1), atol=1e-6), 2,
         "Compute e = torch.exp(x - x.max()) and return e / e.sum().");

## A.2 Temperature

Temperature divides every logit by `T` before softmax. `T < 1` sharpens the distribution (the favorite token gets even more likely), `T > 1` flattens it (rare tokens get a real chance). `T = 0` would divide by zero, so libraries treat it as "always take the most likely token" (greedy decoding). **Entropy** (in bits) summarizes how spread out a distribution is: 0 bits means certain, 1 bit is like a coin flip between two equally likely tokens, and each extra bit doubles the number of equally likely choices.

**TODO A.2:** implement `apply_temperature`.

In [ ]:
def apply_temperature(logits, t):
    """Scale logits by temperature t > 0."""
    # TODO: return logits / t
    return logits


def entropy_bits(logits):
    """Entropy in bits of softmax(logits). (Provided.)"""
    p = torch.softmax(logits, dim=-1)
    p = p[p > 0]
    return float(-(p * torch.log2(p)).sum())


temp_rows = []
for t in [0.3, 0.7, 1.0, 1.5]:
    scaled = apply_temperature(logits, t)
    probs = torch.softmax(scaled, -1)
    top = torch.topk(probs, 3)
    temp_rows.append({"T": t, "top1_prob": round(top.values[0].item(), 3),
                      "top1": tokenizer.decode([int(top.indices[0])]), "top3_probs": [round(v, 3) for v in top.values.tolist()],
                      "entropy_bits": round(entropy_bits(scaled), 2)})
temp_table = pd.DataFrame(temp_rows)
temp_table

**Reading the table:** one row per temperature `T`. `top1` is the most likely token and `top1_prob` its probability; `top3_probs` are the probabilities of the three most likely tokens; `entropy_bits` is how spread out the whole distribution is. Watch `top1_prob` fall and `entropy_bits` rise as `T` goes up.

In [ ]:
ck.check("A2_temperature_function", torch.allclose(apply_temperature(logits, 0.5), logits * 2), 2,
         "apply_temperature should return logits / t.")
ck.check("A2_entropy_rises_with_T", temp_table["entropy_bits"].iloc[0] < temp_table["entropy_bits"].iloc[-1], 1,
         "Entropy at T=0.3 should be lower than at T=1.5. Re-run the table cell after fixing apply_temperature.")
ck.record("entropy_T0.3", float(temp_table["entropy_bits"].iloc[0]))
ck.record("entropy_T1.5", float(temp_table["entropy_bits"].iloc[-1]));

## A.3 Top-k

Top-k keeps only the `k` highest logits and sets the rest to `-inf` (minus infinity), so after softmax they have probability 0. In the code, `masked_fill(condition, value)` replaces every entry where the condition is true with `value`. Chapter 2 (**"Top-k"**) presents it mainly as a way to reduce the softmax computation without giving up too much response diversity. It also cuts off the long tail of very unlikely tokens (the many thousands of tokens that each have a tiny probability).

**TODO A.3:** finish `top_k_filter`.

In [ ]:
def top_k_filter(logits, k):
    """Keep the k largest logits; set the others to -inf. k <= 0 means 'no top-k'."""
    if k is None or k <= 0:
        return logits
    # TODO: kth = the k-th largest value, torch.topk(logits, k).values[-1]
    kth = None
    if kth is None:
        return logits
    # TODO: return logits.masked_fill(logits < kth, float("-inf"))
    return logits


k_out = top_k_filter(logits, 20)
print("tokens left after top-k=20:", int(torch.isfinite(k_out).sum()))

In [ ]:
ck.check("A3_top_k", int(torch.isfinite(top_k_filter(logits, 20)).sum()) == 20
         and int(torch.isfinite(top_k_filter(logits, 1)).sum()) == 1
         and torch.equal(top_k_filter(logits, 0), logits), 2,
         "Find kth = torch.topk(logits, k).values[-1], then masked_fill everything smaller with -inf.");

## A.4 Top-p (nucleus sampling)

Top-p keeps the *smallest* set of most-likely tokens whose probabilities add up to at least `p`. Unlike top-k, the number of tokens kept adapts: when the model is confident, very few tokens survive; when it is unsure, many do (Chapter 2, **"Sampling Strategies"**).

The trick: sort tokens from most to least likely and walk down the list. A token is removed if the tokens ranked *above* it already reach `p`.

**Worked example.** Five tokens, already sorted, with probabilities `[0.5, 0.2, 0.15, 0.1, 0.05]` and `p = 0.8`:

| token | prob | `cum` (running total) | `cum - prob` (mass of the tokens above it) | removed? (`cum - prob >= 0.8`) |
|---|---|---|---|---|
| 1st | 0.50 | 0.50 | 0.00 | no |
| 2nd | 0.20 | 0.70 | 0.50 | no |
| 3rd | 0.15 | 0.85 | 0.70 | no |
| 4th | 0.10 | 0.95 | 0.85 | **yes** |
| 5th | 0.05 | 1.00 | 0.95 | **yes** |

The first three tokens already add up to 0.85, which reaches 0.8, so everything below them is cut. Using `cum - prob` (instead of `cum`) is what keeps the token that *crosses* the line: the 3rd token's own probability pushes the total past 0.8, but the tokens above it only had 0.70.

**Your turn (on paper first):** with the same five probabilities and `p = 0.6`, how many tokens are kept? Put the number in `kept_in_example` in the next cell.

**TODO A.4:** write the one line that decides which sorted tokens to remove.

In [ ]:
kept_in_example = None   # TODO: how many of the five example tokens survive top-p with p = 0.6?


def top_p_filter(logits, p):
    """Nucleus filter: keep the smallest set of top tokens with total probability >= p."""
    if p is None or p >= 1.0:
        return logits
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = torch.softmax(sorted_logits, dim=-1)
    cum = torch.cumsum(probs, dim=-1)            # running total, most likely first
    # TODO: remove_sorted = (cum - probs) >= p   (mass of the tokens ranked above this one already reaches p)
    remove_sorted = torch.zeros_like(probs, dtype=torch.bool)
    remove = torch.zeros_like(remove_sorted).scatter(0, sorted_idx, remove_sorted)   # back to vocabulary order
    return logits.masked_fill(remove, float("-inf"))


for p in [0.5, 0.8, 0.95]:
    print(f"top-p={p}: {int(torch.isfinite(top_p_filter(logits, p)).sum())} tokens kept")

In [ ]:
def _nucleus_ok(lg, p):
    full = torch.softmax(lg, -1)
    kept = torch.isfinite(top_p_filter(lg, p))
    mass = full[kept].sum().item()
    smallest = full[kept].min().item()
    return kept.sum() >= 1 and mass >= p - 1e-6 and (mass - smallest) < p + 1e-6


flat = apply_temperature(logits, 1.5)
ck.check("A4_top_p", _nucleus_ok(logits, 0.8) and _nucleus_ok(flat, 0.9)
         and int(torch.isfinite(top_p_filter(flat, 0.9)).sum()) < logits.numel(), 3,
         "The kept tokens must add up to at least p, and dropping the least likely kept token must fall below p. "
         "Use remove_sorted = (cum - probs) >= p.")
ck.check("A4_worked_example", kept_in_example == 2, 1,
         "With p = 0.6: cum - prob is 0.00, 0.50, 0.70, ... so which tokens have cum - prob >= 0.6?");

## A.5 Prove it matches `generate()`

Why bother? Every model API you'll use (including the course cluster) has `temperature`, `top_k`, and `top_p` settings. Matching `generate()` proves those knobs do exactly what you just built, so you'll know what you're changing when you set them. Hugging Face's `generate()` applies the same steps in the same order: temperature, then top-k, then top-p, then it samples. With `output_logits=True` it returns the raw logits it saw, and with `output_scores=True` the processed scores after those steps. If your functions are right, your processed logits keep the same tokens with the same probabilities. (When the distribution is very flat and hundreds of tokens survive top-p, rounding can move the cutoff by one token; the check allows a difference of up to 2 tokens.)

In [ ]:
def my_process(logits, temperature, top_k, top_p):
    x = apply_temperature(logits.float(), temperature)
    x = top_k_filter(x, top_k)
    return top_p_filter(x, top_p)


inputs = chat_inputs(PROMPT)
compare_rows = []
for (t, k, p) in [(0.7, 20, 0.8), (1.0, 0, 0.9), (1.3, 50, 1.0), (0.6, 20, 0.95)]:
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=1, do_sample=True, temperature=t, top_k=k, top_p=p,
                             output_scores=True, output_logits=True, return_dict_in_generate=True,
                             pad_token_id=tokenizer.eos_token_id)
    raw, hf = out.logits[0][0].float(), out.scores[0][0].float()
    mine = my_process(raw, t, k, p)
    same = torch.equal(torch.isfinite(mine), torch.isfinite(hf))
    n_different = int((torch.isfinite(mine) ^ torch.isfinite(hf)).sum())
    pm = my_softmax(mine) if my_softmax(mine) is not None else torch.softmax(mine, -1)
    diff = (pm - torch.softmax(hf, -1)).abs().max().item()
    my_pick = tokenizer.decode([int(torch.multinomial(pm, 1))])     # sample one token with YOUR sampler
    compare_rows.append({"T": t, "top_k": k, "top_p": p, "tokens_kept_hf": int(torch.isfinite(hf).sum()),
                         "tokens_kept_mine": int(torch.isfinite(mine).sum()), "same_set": same, "tokens_different": n_different,
                         "max_prob_diff": diff, "my_sample": repr(my_pick)})
compare = pd.DataFrame(compare_rows)
compare

**Reading the table:** one row per sampling setting. `T`, `top_k`, and `top_p` are the settings (`top_k=0` and `top_p=1.0` mean that filter is off). `tokens_kept_hf` and `tokens_kept_mine` count the tokens still allowed after filtering by Hugging Face and by your code; `same_set` is `True` when both kept exactly the same tokens, and `tokens_different` counts any that differ. `max_prob_diff` is the largest difference in any token's probability (a tiny number such as `1e-07` is just rounding). `my_sample` is one token drawn with your sampler.

In [ ]:
ck.check("A5_matches_generate", compare["tokens_different"].max() <= 2 and compare["max_prob_diff"].max() < 1e-3, 3,
         "Your sampler keeps different tokens than generate(). Look at the row where same_set is False and "
         "re-check that step (temperature, top-k, or top-p).");

# Part B - Applied lab: what sampling does to Larkspur answers

The helper below samples `n` answers to the same prompt in one **batched** call (`num_return_sequences`: the answers are generated together, which is faster than one at a time). `temperature=0` means greedy, which gives the same output every time, so it runs once and repeats the result. We set `top_k=0, top_p=1.0` so that **only temperature** changes between runs.

In [ ]:
def sample_n(messages, n, temperature, max_new_tokens=40, top_k=0, top_p=1.0, mdl=None, tok=None):
    mdl, tok = mdl or model, tok or tokenizer
    inputs = chat_inputs(messages, tok, mdl)
    L = inputs["input_ids"].shape[1]
    with torch.no_grad():
        if temperature == 0:
            out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tok.eos_token_id)
            return [tok.decode(out[0][L:], skip_special_tokens=True).strip()] * n
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                           top_k=top_k, top_p=top_p, num_return_sequences=n, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[L:], skip_special_tokens=True).strip() for o in out]


print(sample_n(PROMPT, 3, 0.7, max_new_tokens=8))

## B.1 Diversity versus temperature

A common way to measure output diversity is **distinct-n**: the number of *unique* n-word sequences divided by the *total* number of n-word sequences across all samples. (An n-word sequence is called an **n-gram**: with n = 2, "reset the password" contains "reset the" and "the password".) 1.0 means no repetition across samples; values near 0 mean the samples say the same thing.

**TODO B.1:** finish `distinct_n`.

In [ ]:
def distinct_n(texts, n=2):
    """Unique n-grams / total n-grams across all texts (lowercased, split on spaces)."""
    ngrams = []
    for t in texts:
        words = t.lower().split()
        # TODO: add every n-gram as a tuple: ngrams += [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
        pass
    # TODO: return len(set(ngrams)) / len(ngrams), or 0.0 if ngrams is empty
    return None


print(distinct_n(["a b c", "a b d"], 2), "(expected 0.75)")

In [ ]:
CREATIVE = [{"role": "user", "content": "Write a one-sentence welcome message for a new Larkspur Ridge IT client."}]
N = 8
div_rows, samples_by_t = [], {}
for t in [0, 0.7, 1.0, 1.5]:
    texts = sample_n(CREATIVE, N, t, max_new_tokens=40)
    samples_by_t[t] = texts
    div_rows.append({"T": t, "distinct_1": distinct_n(texts, 1), "distinct_2": distinct_n(texts, 2),
                     "unique_outputs": len(set(texts))})
    print(f"\n=== T={t}")
    for s in texts[:3]:
        print("  -", s[:150])
diversity = pd.DataFrame(div_rows)
diversity

**Reading the table:** one row per temperature `T` (0 is greedy). `distinct_1` and `distinct_2` are distinct-n for single words and for word pairs across the 8 samples; `unique_outputs` is how many of the 8 answers were not exact copies of another. Above the table you see three sample answers for each temperature.

In [ ]:
ck.check("B1_distinct_n", distinct_n(["a b c", "a b d"], 2) == 0.75 and distinct_n([""], 2) == 0.0, 2,
         "Collect tuples of n consecutive words from every text, then unique/total. Return 0.0 when there are none.")
d2 = diversity["distinct_2"]
ck.check("B1_diversity_grows", d2.notna().all() and d2.iloc[-1] > d2.iloc[0], 2,
         "distinct-2 at T=1.5 should be higher than at T=0 (greedy repeats one answer). Re-run after fixing distinct_n.")
if d2.notna().all():
    for t, v in zip(diversity["T"], d2):
        ck.record(f"distinct2_T{t}", float(v))

## B.2 Consistency and hallucination on Larkspur facts

Two questions, each sampled 10 times at four temperatures (top-k and top-p off):

1. **Grounded:** the answer is in the context we provide (Larkspur's SLA table). There is one right answer: *2 business hours*.
2. **Ungrounded:** we ask for Larkspur's support phone number and give *no* context. Larkspur is fictional, so the model cannot know it. Any specific number it gives is a **hallucination**. (If your model refuses or says it doesn't know instead, that's a result too: report it in Q2 and connect it to post-training, which you'll see in B.3.)

**TODO B.2:** inside the loop, compute the grounded accuracy and the number of distinct ungrounded answers.

In [ ]:
SLA_TABLE = """Larkspur Ridge IT - response targets (Exhibit B, effective 2026-01-01)
| Priority | Essentials | Professional | Premier |
| P1 | 1 hour (business hours only) | 30 minutes (24x7) | 15 minutes (24x7) |
| P2 | 2 business hours | 1 business hour | 1 hour (24x7) |
| P3 | 4 business hours | 4 business hours | 2 business hours |
| P4 | 1 business day | 1 business day | 1 business day |"""
GROUNDED = [{"role": "system", "content": "Answer using only this document:\n" + SLA_TABLE},
            {"role": "user", "content": "What is the response target for a P2 ticket for an Essentials client? "
                                        "Answer in five words or fewer."}]
UNGROUNDED = [{"role": "user", "content": "What is the phone number of Larkspur Ridge IT's support line? "
                                          "Reply with only the phone number."}]


def is_correct_grounded(answer):
    return re.search(r"\b(2|two)\s+business\s+hours?\b", answer.lower()) is not None


def normalize_phone(answer):
    digits = re.sub(r"\D", "", answer)
    return digits if digits else answer.strip().lower()[:40]


cons_rows, examples = [], {}
for t in [0.3, 0.7, 1.0, 1.3]:
    g = sample_n(GROUNDED, 10, t, max_new_tokens=16)
    u = sample_n(UNGROUNDED, 10, t, max_new_tokens=20)
    examples[t] = (g[:3], u[:5])
    # TODO: fraction of g answers where is_correct_grounded(answer) is True
    grounded_accuracy = None
    # TODO: number of distinct normalized ungrounded answers: len({normalize_phone(a) for a in u})
    ungrounded_distinct = None
    cons_rows.append({"T": t, "grounded_accuracy": grounded_accuracy, "ungrounded_distinct": ungrounded_distinct})
consistency = pd.DataFrame(cons_rows)
for t, (g3, u5) in examples.items():
    print(f"T={t}  grounded: {g3}\n        ungrounded: {u5}")
consistency

**Reading the table:** one row per temperature `T`. `grounded_accuracy` is the fraction of the 10 grounded answers that said 2 business hours (1.0 = all correct). `ungrounded_distinct` is how many *different* phone numbers appeared among the 10 ungrounded answers (1 = the model always invented the same one). The printed lines above it show a few raw answers.

In [ ]:
ga, ud = consistency["grounded_accuracy"], consistency["ungrounded_distinct"]
ck.check("B2_consistency_table", len(consistency) == 4 and ga.notna().all() and ga.between(0, 1).all()
         and ud.notna().all() and (ud >= 1).all(), 3,
         "grounded_accuracy = sum(is_correct_grounded(a) for a in g) / len(g); "
         "ungrounded_distinct = len({normalize_phone(a) for a in u}).")
if ga.notna().all() and ud.notna().all():
    ck.record("grounded_acc_T0.3", float(ga.iloc[0]))
    ck.record("grounded_acc_T1.3", float(ga.iloc[-1]))
    ck.record("ungrounded_distinct_T0.3", int(ud.iloc[0]))
    ck.record("ungrounded_distinct_T1.3", int(ud.iloc[-1]))

## B.3 Base model versus post-trained model

Chapter 2 (**"Post-Training"**) describes two steps that turn a pretrained model into a chat assistant: **supervised finetuning (SFT)** on demonstration conversations, then **preference finetuning** (RLHF, DPO and similar) to make answers more helpful and safe. The base model only learned to continue text. Load the base model and give both models the same three requests: the base model gets the plain text, the instruct model gets it through its chat template (the special-token format that marks the user's turn).

In [ ]:
BASE_ID = "Qwen/Qwen3-1.7B-Base"
base_tok = AutoTokenizer.from_pretrained(BASE_ID)
base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, dtype=DTYPE, device_map="auto")
base_model.eval()
if gpu:
    print(f"GPU memory in use with both models: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def base_continue(text, max_new_tokens=80):
    enc = base_tok(text, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=base_tok.eos_token_id)
    return base_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


REQUESTS = ["How do I reset my Windows password?",
            "Write a two-sentence apology to a client for missing a response deadline.",
            "What is 17 * 23?"]
pairs = []
for r in REQUESTS:
    b = base_continue(r + "\n")
    i = sample_n([{"role": "user", "content": r}], 1, 0, max_new_tokens=80)[0]
    pairs.append((r, b, i))
    print(f"##### {r}\n--- BASE (plain text continuation):\n{b}\n--- INSTRUCT (chat template):\n{i}\n")

Post-training may also change the *shape* of the distribution: a post-trained model is often more confident (lower entropy) about what comes next. This is the lab's claim to test, not a statement from the chapter, and with only three prompts **your numbers can go either way: report what you see.** We measure both models on the same plain-text prompts; the cell also prints (not graded) the instruct model's entropy when the prompt goes through its chat template, the way it is normally used.

**TODO B.3:** compute the mean next-token entropy for each model across the three requests, using `entropy_bits` from A.2. The cell prints each model's three entropies (one per request, in the order of `REQUESTS`) and then the mean.

In [ ]:
def raw_next_logits(mdl, tok, text):
    enc = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        return mdl(**enc).logits[0, -1, :].float()


base_entropies = [entropy_bits(raw_next_logits(base_model, base_tok, r + "\n")) for r in REQUESTS]
instruct_entropies = [entropy_bits(raw_next_logits(model, tokenizer, r + "\n")) for r in REQUESTS]

# TODO: the mean of each list (sum(...) / len(...))
mean_entropy_base = None
mean_entropy_instruct = None
print("base:", [round(x, 2) for x in base_entropies], "mean", mean_entropy_base)
print("instruct:", [round(x, 2) for x in instruct_entropies], "mean", mean_entropy_instruct)
instruct_chat_entropies = [entropy_bits(next_logits([{"role": "user", "content": r}])) for r in REQUESTS]
print("instruct, through its chat template (not graded):", [round(x, 2) for x in instruct_chat_entropies])

**TODO B.3 (continued):** in one or two sentences each, describe what you observed. These short notes feed analysis Q3.

In [ ]:
observations = {
    "format": "",        # TODO: how did the base model's output format differ from the instruct model's?
    "instructions": "",  # TODO: did the base model follow the request (e.g., two sentences)? did the instruct model?
    "math": "",          # TODO: who got 17 * 23 = 391 right, and how did each present it?
}

In [ ]:
ok_ent = mean_entropy_base is not None and mean_entropy_instruct is not None
ck.check("B3_entropies", ok_ent and abs(mean_entropy_base - sum(base_entropies) / 3) < 1e-6
         and abs(mean_entropy_instruct - sum(instruct_entropies) / 3) < 1e-6, 2,
         "mean_entropy_base = sum(base_entropies) / len(base_entropies); same for instruct.")
ck.check("B3_observations", all(len(observations.get(k, "").strip()) >= 20 for k in ["format", "instructions", "math"]), 2,
         "Write at least a full sentence for each of the three observations.")
if ok_ent:
    ck.record("mean_entropy_base", float(mean_entropy_base))
    ck.record("mean_entropy_instruct", float(mean_entropy_instruct))

# Written analysis

Answer in the box under each question (3-6 sentences). Use **your own numbers**, and cite Chapter 2 by section name. Q1 is **optional (not graded)**; Q2 and Q3 here plus Q4-Q6 in `lab03_cluster.ipynb` are the graded questions, and the AI-use reflection is in the cluster notebook too.

**Q1 (optional, not graded). Temperature, top-k, top-p.** Using your A.2 table and A.5 comparison, explain what temperature does to the distribution and how top-k and top-p differ (how many tokens did each keep for your prompt?). Cite Chapter 2's **"Sampling Strategies"**. Qwen recommends temperature 0.7 with top-p 0.8 and top-k 20 for this model in non-thinking mode: why combine them instead of using temperature alone?

*Your answer:*

**Q2. Inconsistency and hallucination.** Report your B.2 table: grounded accuracy and distinct ungrounded answers at each temperature. Using Chapter 2's **"The Probabilistic Nature of AI"** (inconsistency and hallucination), explain why the model gave confident, *different* phone numbers when it had no context, and why grounding helped. Does lowering the temperature fix hallucination, or only hide it? Use your numbers.

*Your answer:*

**Q3. What post-training adds.** Using your B.3 outputs, observations, and mean entropies, describe how the base and instruct models differed. Which differences come from **supervised finetuning** and which from **preference finetuning**, according to Chapter 2's **"Post-Training"** section? (Reminder: the chapter says SFT trains on high-quality instruction data so the model answers conversations instead of just continuing text, and preference finetuning further shapes responses to match what people prefer. Of your three requests, the apology is the one most likely to show preference finetuning: tone and politeness.) Finally, from your own entropy numbers (this part is the lab's idea, not the chapter's): if the instruct model was more confident, why could that be useful for Larkspur's assistant and a cost for creative or diverse outputs? If it wasn't, say so and suggest why.

*Your answer:*

## Save your progress
Run this last, save (**File > Download > Download .ipynb**), and upload the .ipynb to Canvas together with `lab03_cluster.ipynb`.

In [ ]:
ck.report()